# Exploración de datos (EDA) — Titanic

Pregunta central: ¿qué factores tuvieron más influencia sobre la supervivencia de los pasajeros del Titanic?

El cuaderno solo **lee** los datos de `data/` (generados con `make ingest`, `make silver` y `make gold`). La lógica de las transformaciones vive en `flows/`.

## 1. Diagnóstico sobre Bronze

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

from flows.common.paths import BRONZE, BRONZE_LABELED_NAME, GOLD, SILVER, SILVER_NAME

sns.set_theme(style='whitegrid', context='notebook')
plt.rcParams['figure.figsize'] = (8, 5)
plt.rcParams['axes.titlesize'] = 13

# Cargamos Bronze (la ingesta ya guardó los datos)
df = pd.read_csv(BRONZE / BRONZE_LABELED_NAME)

### Paso 1. Dimensiones

In [ ]:
df.shape

**Resultado:** 891 pasajeros (filas) y 12 variables (columnas).

### Paso 2. Variables, nombres y tipos

In [ ]:
df.info()

**Resultado:** `Age` (714 no nulos), `Cabin` (204) y `Embarked` (889) tienen menos valores que las 891 filas, así que ahí hay datos faltantes.

### Paso 3. Primeras 10 filas

In [ ]:
df.head(10)

### Paso 4. Estadísticas descriptivas

In [ ]:
df.describe()

In [ ]:
df.select_dtypes(exclude='number').describe()

**Resultado:** la edad promedio es 29.7 años y la tarifa promedio 32.2, pero la mediana de la tarifa es 14.45: hay tarifas muy altas (máximo 512.33). En las variables de texto hay 891 nombres distintos, 681 tiquetes y 147 cabinas distintas.

### Paso 5. Datos faltantes

In [ ]:
df.isnull().sum()

In [ ]:
sns.heatmap(df.isnull(), cbar=False, yticklabels=False, cmap='viridis')
plt.title('Mapa de datos faltantes (amarillo = falta el dato)')
plt.show()

**Resultado:** faltan 177 edades, 687 cabinas y 2 puertos de embarque; son las únicas columnas con faltantes. `Cabin` falta en el 77 % de las filas, `Age` en el 20 % y `Embarked` en menos del 1 %.

## 2. Exploración sobre Silver

Desde aquí se lee la capa Silver.

In [ ]:
bronze = df
df = pd.read_parquet(SILVER / SILVER_NAME)
df.shape

### Paso 6. Género y clase

In [ ]:
sns.countplot(data=df, x='Sex')
plt.title('Pasajeros por sexo')
plt.show()

In [ ]:
sns.countplot(data=df, x='Sex', hue='Pclass')
plt.title('Pasajeros por sexo y clase')
plt.show()

In [ ]:
pd.crosstab(df['Sex'], df['Pclass'], normalize='columns')

**Resultado:** hay más hombres que mujeres (577 vs. 314). Las mujeres son el 43.5 % de la primera clase y el 29.3 % de la tercera.

### Paso 7. Distribución de edades

`Age` en Silver ya está imputada; para ver solo las edades reales se usa `AgeOriginal`.

In [ ]:
sns.histplot(df['AgeOriginal'].dropna(), bins=30)
plt.title('Distribución de la edad (edades conocidas)')
plt.xlabel('Edad')
plt.ylabel('Pasajeros')
plt.show()

In [ ]:
edad = df['AgeOriginal']
print('Promedio:', round(edad.mean(), 2))
print('Mediana:', edad.median())
print('Mínimo:', edad.min())
print('Máximo:', edad.max())

**Resultado:** la edad se concentra entre los 20 y los 40 años (385 de 714 pasajeros con edad). Promedio 29.7, mediana 28, mínimo 0.42 y máximo 80.

### Paso 8. Variable `Person`

In [ ]:
df['Person'].value_counts()

**Resultado:** 537 hombres, 271 mujeres y 83 niños (menores de 16). Los 177 pasajeros sin edad quedan clasificados por su sexo, no como niños.

### Paso 9. Niños por clase

In [ ]:
pd.crosstab(df['Pclass'], df['Person'])

**Resultado:** 58 de los 83 niños viajaban en tercera clase; solo 6 en primera.

### Paso 10. Densidad (KDE) de la edad

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4), sharey=True)
sns.kdeplot(data=df, x='AgeOriginal', hue='Sex', fill=True, common_norm=False, ax=axes[0])
sns.kdeplot(data=df, x='AgeOriginal', hue='Person', fill=True, common_norm=False, ax=axes[1])
sns.kdeplot(data=df, x='AgeOriginal', hue='Pclass', fill=True, common_norm=False,
            palette='viridis', ax=axes[2])
axes[0].set_title('Edad por sexo')
axes[1].set_title('Edad por tipo de persona')
axes[2].set_title('Edad por clase')
plt.tight_layout()
plt.show()

**Resultado:** los pasajeros de primera clase tienen edades más altas (mediana 37) que los de tercera (mediana 24).

### Paso 11. Cubierta de la cabina

In [ ]:
df['TCabin'].value_counts()

### Paso 12. Conteo por cubierta

In [ ]:
cubierta = df.loc[df['TCabin'] != 'X', 'TCabin']
sns.countplot(x=cubierta, order=sorted(cubierta.unique()))
plt.title('Pasajeros por cubierta (cabinas conocidas)')
plt.xlabel('Cubierta')
plt.ylabel('Pasajeros')
plt.show()

In [ ]:
# La cubierta 'T' tiene un solo registro: se excluye solo del gráfico
cubierta = cubierta[cubierta != 'T']
sns.countplot(x=cubierta, order=sorted(cubierta.unique()))
plt.title('Pasajeros por cubierta (sin la cubierta T)')
plt.xlabel('Cubierta')
plt.ylabel('Pasajeros')
plt.show()

**Resultado:** las cubiertas con más cabinas conocidas son C (59), B (47), D (33) y E (32). La cubierta T tiene un solo registro, por eso se excluye solo del gráfico.

### Paso 13. Puerto de embarque y clase

In [ ]:
sns.countplot(data=df, x='Embarked', hue='Pclass')
plt.title('Pasajeros por puerto de embarque y clase')
plt.show()

In [ ]:
pd.crosstab(df['Embarked'], df['Pclass'], normalize='index')

**Resultado:** Cherbourg tiene una proporción alta de primera clase (50.6 %), mientras que Queenstown es casi toda tercera clase (93.5 %).

### Paso 14. Variable `Alone`

In [ ]:
df['Alone'].value_counts(normalize=True)

### Paso 15. Distribución de `Alone`

In [ ]:
sns.countplot(data=df, x='Alone')
plt.title('Pasajeros solos y con familia')
plt.xlabel('Viaja')
plt.ylabel('Pasajeros')
plt.show()

## 3. Supervivencia

### Paso 16. Panorama general

In [ ]:
sns.countplot(data=df, x='Survivor')
plt.title('Sobrevivientes')
plt.show()

In [ ]:
df['Survived'].mean()

**Resultado:** sobrevivió el 38.4 % de los pasajeros (342 de 891).

### Paso 17. Supervivencia por clase

In [ ]:
sns.catplot(data=df, x='Pclass', y='Survived', kind='point', height=4, aspect=1.4)
plt.title('Supervivencia por clase')
plt.ylabel('Tasa de supervivencia')
plt.show()

**Resultado:** la supervivencia baja con la clase: 63.0 % en primera, 47.3 % en segunda y 24.2 % en tercera.

### Paso 18. Supervivencia por clase y tipo de persona

In [ ]:
sns.catplot(data=df, x='Pclass', y='Survived', hue='Person', hue_order=['child', 'female', 'male'],
            kind='point', height=4, aspect=1.4)
plt.title('Supervivencia por clase y tipo de persona')
plt.ylabel('Tasa de supervivencia')
plt.show()

**Resultado:** dentro de cada clase las mujeres sobreviven mucho más que los hombres (en primera 98 % vs. 35 %). Los niños de segunda clase sobrevivieron todos (19 niños) y los de tercera solo el 43 %.

### Paso 19. Edad y supervivencia

In [ ]:
sns.lmplot(data=df, x='AgeOriginal', y='Survived')
plt.show()

In [ ]:
sns.lmplot(data=df, x='AgeOriginal', y='Survived', hue='Pclass')
plt.show()

In [ ]:
generaciones = [10, 20, 40, 60, 80]
sns.lmplot(data=df, x='AgeOriginal', y='Survived', hue='Pclass', x_bins=generaciones)
plt.show()

In [ ]:
sns.lmplot(data=df, x='AgeOriginal', y='Survived', hue='Sex', x_bins=generaciones)
plt.show()

**Resultado:** los menores de 10 años tienen mayor supervivencia (59 %) y los mayores de 60 la menor (23 %). Por sexo, la edad casi no cambia a las mujeres hasta los 60 años (entre 61 % y 79 %), pero los hombres adultos sobreviven poco (14 % a 19 %).

### Paso 20. Relación tarifa-edad

In [ ]:
plt.scatter(df['AgeOriginal'], df['Fare'], alpha=0.5)
plt.title('Tarifa vs. edad')
plt.xlabel('Edad')
plt.ylabel('Tarifa')
plt.show()

In [ ]:
sns.scatterplot(data=df, x='AgeOriginal', y='Fare', hue='Pclass', alpha=0.5, palette='viridis')
plt.title('Tarifa vs. edad por clase')
plt.xlabel('Edad')
plt.ylabel('Tarifa')
plt.show()

**Resultado:** la relación entre edad y tarifa es débil (correlación 0.10). Las tarifas más altas son de primera clase.

## 4. Limpieza y preparación

### Paso 21. Faltantes en `Embarked`

In [ ]:
print('Faltantes en Bronze:', bronze['Embarked'].isna().sum())
print('Moda:', bronze['Embarked'].mode()[0])
print('Faltantes en Silver:', df['Embarked'].isna().sum())
df[['Embarked', 'NEmbarked']].drop_duplicates()

**Resultado:** faltaban 2 valores y se rellenaron con la moda, `S` (Southampton). Es una imputación razonable porque solo son 2 de 891 filas.

### Paso 22. Faltantes en `Age`

In [ ]:
print('Desviación estándar antes:', round(df['AgeOriginal'].std(), 2))
print('Desviación estándar después:', round(df['Age'].std(), 2))
print('Edades imputadas:', df['AgeImputada'].sum())

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharey=True)
sns.histplot(df['AgeOriginal'].dropna(), bins=30, ax=axes[0])
sns.histplot(df['Age'], bins=30, ax=axes[1])
axes[0].set_title('Edad original')
axes[1].set_title('Edad imputada')
plt.tight_layout()
plt.show()

**Resultado:** se imputaron 177 edades y la desviación estándar bajó de 14.53 a 13.01. Baja porque todas las edades imputadas toman el mismo valor (el promedio de su sexo), lo que agrega un pico en el histograma alrededor de ese promedio.

### Paso 23. `Sex` a número

In [ ]:
df[['Sex', 'NSex']].drop_duplicates()

### Paso 24. Tratamiento de `Cabin`

In [ ]:
df[['TCabin', 'NCabin']].drop_duplicates().sort_values('NCabin')

### Paso 25. Títulos del nombre

In [ ]:
titulos = df['Name'].str.extract(r',\s*([^.]+)\.', expand=False).str.strip()
titulos.value_counts()

In [ ]:
pd.crosstab(titulos, df['Sex'])

### Paso 26. Títulos consolidados

In [ ]:
df['TTitulo'].value_counts()

**Resultado:** quedaron 4 grupos: Mr (557), Miss (186), Mrs (127) y Otro (21).

## 5. Correlaciones y estadísticas por grupos

### Paso 27. Matriz de correlación

In [ ]:
cols = ['Survived', 'Pclass', 'NSex', 'Age', 'SibSp', 'Parch', 'Fare', 'NEmbarked', 'NCabin',
        'NTitulo']
corr = df[cols].corr()
plt.figure(figsize=(9, 7))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', center=0, vmin=-1, vmax=1)
plt.title('Matriz de correlación')
plt.show()

In [ ]:
corr['Survived'].sort_values()

**Resultado:** las variables más asociadas linealmente con `Survived` son `NSex` (-0.54), `NTitulo` (0.43), `Pclass` (-0.34) y `NCabin` (-0.30). Las variables categóricas codificadas con un orden arbitrario (`NSex`, `NTitulo`, `NCabin`) hay que leerlas con cuidado.

### Paso 28. Estadísticas por grupos (tablas Gold)

In [ ]:
for nombre in ['age_by_survival', 'survival_by_sex', 'survival_by_pclass', 'survival_by_alone']:
    print(nombre)
    display(pd.read_parquet(GOLD / f'{nombre}.parquet'))

**Resultado:** sobrevivió el 74.2 % de las mujeres y el 18.9 % de los hombres. Quienes viajaban con familia sobrevivieron más (50.6 %) que quienes viajaban solos (30.4 %).

### Paso 29. Chi-cuadrado: sexo y supervivencia

In [ ]:
import json

json.loads((GOLD / 'chi2_sex_survival.json').read_text())

**Resultado:** chi2 = 260.7 con p-valor ≈ 1.2e-58, así que se rechaza la independencia entre sexo y supervivencia. Esto indica que hay una asociación muy fuerte, pero no dice que el sexo sea la causa ni qué tan grande es el efecto por sí solo.

## 6. Visualizaciones comparativas

### Paso 30. Boxplots de la edad

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 9))
sns.boxplot(data=df, x='Sex', y='AgeOriginal', ax=axes[0, 0])
sns.boxplot(data=df, x='Pclass', y='AgeOriginal', ax=axes[0, 1])
sns.boxplot(data=df, x='Embarked', y='AgeOriginal', ax=axes[1, 0])
sns.boxplot(data=df, x='Survived', y='AgeOriginal', ax=axes[1, 1])
plt.tight_layout()
plt.show()

**Resultado:** las medianas de edad por clase son 37, 29 y 24 (de primera a tercera); por sexo y por supervivencia son casi iguales (27 y 29; 28 y 28).

### Paso 31. Distribución de la tarifa

In [ ]:
sns.histplot(df['Fare'], kde=True, bins=40)
plt.title('Distribución de la tarifa')
plt.show()

In [ ]:
print('Media:', round(df['Fare'].mean(), 2))
print('Mediana:', round(df['Fare'].median(), 2))
print('Asimetría:', round(df['Fare'].skew(), 2))

**Resultado:** la media de la tarifa (32.2) es muy superior a la mediana (14.45) y la asimetría es 4.79: la distribución está muy sesgada a la derecha. Esto afecta a los modelos lineales, que son sensibles a valores extremos.

## 7. Dataset final

### Pasos 32 y 33. Limpieza final y verificación

In [ ]:
from flows.common.paths import SILVER_FINAL_NAME

final = pd.read_parquet(SILVER / SILVER_FINAL_NAME)
print(final.shape)
print(final.columns.tolist())
final.drop(columns='AgeOriginal').isna().sum().sum()

**Resultado:** el dataset final tiene 891 filas y 21 columnas, sin nulos (salvo `AgeOriginal`, que conserva los faltantes a propósito).

## 8. Análisis crítico

### Paso 34. Clase × género

In [ ]:
pd.read_parquet(GOLD / 'survival_by_class_sex.parquet')

**Resultado:** el género pesa más que la clase en primera y segunda (98 % y 92 % de mujeres sobreviven frente a 37 % y 16 % de hombres). En tercera clase la brecha es menor: 50 % de las mujeres y 13.5 % de los hombres.

### Paso 35. Puerto de embarque y supervivencia

In [ ]:
pd.read_parquet(GOLD / 'survival_by_embarked.parquet')

In [ ]:
pd.read_parquet(GOLD / 'survival_by_embarked_class.parquet')

**Resultado:** Cherbourg parece tener más supervivencia (55.4 %) que Southampton (33.9 %). Al separar por clase la diferencia se reduce: en primera clase es 69 % (C) vs. 59 % (S), y en tercera 38 % vs. 19 %. Parte del efecto del puerto es una confusión con la clase.

### Paso 36. Tarifa dentro de cada clase

In [ ]:
pd.read_parquet(GOLD / 'survival_by_fare_quartile_class.parquet')

**Resultado:** en primera y segunda clase, pagar más se asocia con mayor supervivencia (en primera pasa de 46 % a 74 %). En tercera clase la relación es débil (23 % a 28 %).

### Paso 37. Tamaño de la familia

In [ ]:
pd.read_parquet(GOLD / 'survival_by_family.parquet')

In [ ]:
sns.pointplot(data=df, x='Familia', y='Survived')
plt.title('Supervivencia por tamaño de la familia')
plt.show()

**Resultado:** las familias de 1 a 3 familiares sobreviven más (55 % a 72 %) que quienes viajan solos (30 %). En familias de 6 o más hay muy pocos pasajeros (12, 6 y 7) y los intervalos de confianza son muy anchos, así que esos porcentajes son poco confiables.

### Paso 38. Pasajeros atípicos

In [ ]:
pd.read_parquet(GOLD / 'outlier_summary.parquet')

In [ ]:
df.nlargest(10, 'Fare')[['Name', 'Pclass', 'Fare', 'Survived']]

**Resultado:** hay 116 atípicos en `Fare`, 11 en la edad original y 91 en `Familia`. Los atípicos de `Fare` sobreviven más (68.1 %) que el resto (33.9 %); son pasajeros de primera clase, es decir, casos reales y no errores.